# SAMap broad cell-type analysis

This analysis aligns comparable broad intestinal cell classes across species:
**Fibroblasts, SMCs, Endothelial, and Epithelial cells where present**.

The public notebook includes:
- combined and species-separated SAMap UMAPs;
- combined and species-separated cell-type UMAPs;
- checking whether a gene of interest is present in each species and the exact feature name;
- separate-panel gene-expression plots using the original normalized species AnnData objects;
- a combined gene-expression SAMap UMAP with within-species 0–1 scaling;
- cell-type mapping scores;
- fibroblast GenePairFinder / BLAST tables;
- fibroblast gene-network, UpSet, and paralog-substitution workflows.

Project-specific marker identities and unpublished biological conclusions are not embedded.


## Expected inputs and outputs

**Inputs**
- one normalized `.h5ad` file per species in `outputs/samap/inputs/celltype/`;
- broad cell-type metadata for Fibroblasts, SMCs, Endothelial, and Epithelial cells where present;
- pairwise SAMap BLAST maps in `maps/`;

**Outputs**
- integrated SAMap AnnData in `outputs/samap/celltype/objects/`;
- figures in `outputs/samap/celltype/figures/`;
- mapping, GenePairFinder, network, and species-combination tables in `outputs/samap/celltype/tables/`;
- run parameters and package versions in `outputs/samap/celltype/metadata/`.


In [ ]:
from pathlib import Path

import pandas as pd
import scanpy as sc
from threadpoolctl import threadpool_limits

from samap import SAMAP
from samap.sam import SAM
from samap.analysis import GenePairFinder

from python.samap.samap_analysis_utils import (
    load_samap_config,
    species_files_from_config,
    result_directories_from_config,
    validate_h5ad_inputs,
    validate_blast_maps,
    validate_integrated_metadata,
    save_run_metadata,
    SPECIES_ORDER,
    SPECIES_LABELS,
    combine_species_metadata,
    get_celltype_mapping_scores,
    plot_samap_species,
    plot_samap_species_separate,
    plot_samap_celltypes_combined,
    plot_samap_celltypes_separate_species,
    check_gene_presence_across_species,
    check_gene_map_presence,
    plot_gene_across_species_panels,
    plot_gene_combined_relative_expression,
    run_all_pairwise_gene_pairs,
    build_gene_pair_network,
    summarize_connected_components,
    add_species_combo,
    save_components_by_species_combo,
    plot_species_upset,
    plot_gene_network_component,
    run_paralog_substitutions,
)


## Function reference used in this notebook

| Function | What it does |
|---|---|
| `combine_species_metadata()` | Merges species-specific cell-type annotation columns into one integrated SAMap metadata column. |
| `get_celltype_mapping_scores()` | Calculates SAMap mapping scores between annotated cell populations across species. |
| `plot_samap_species()` | Plots all species together on the shared SAMap embedding. |
| `plot_samap_species_separate()` | Shows each species in a separate panel while keeping the same integrated SAMap coordinates. |
| `plot_samap_celltypes_combined()` | Colors cells by broad cell type and distinguishes species on one shared SAMap UMAP. |
| `plot_samap_celltypes_separate_species()` | Shows broad cell-type annotations separately for each species using the same integrated coordinates. |
| `check_gene_presence_across_species()` | Checks whether a gene is present in each species and reports the exact stored feature name. |
| `check_gene_map_presence()` | Verifies a manually curated homolog/paralog mapping against the source AnnData objects. |
| `plot_gene_across_species_panels()` | Plots one homolog/paralog group in separate species panels using original normalized expression and shared SAMap coordinates. |
| `plot_gene_combined_relative_expression()` | Combines homolog/paralog expression on one SAMap UMAP after within-species 0–1 relative-expression scaling. |
| `run_all_pairwise_gene_pairs()` | Runs fibroblast GenePairFinder analysis for every species pair and joins BLAST sequence statistics. |
| `build_gene_pair_network()` | Converts pairwise fibroblast gene relationships into a NetworkX graph. |
| `summarize_connected_components()` | Identifies connected gene-network components and summarizes which species occur in each one. |
| `add_species_combo()` | Creates an exact species-combination label such as `hu+mo+ze` for every connected component. |
| `save_components_by_species_combo()` | Writes separate CSV files for components with each exact species combination. |
| `plot_species_upset()` | Summarizes the frequency of exact species combinations across gene-network components with an UpSet plot. |
| `plot_gene_network_component()` | Draws one selected connected gene-network component. |
| `run_paralog_substitutions()` | Runs SAMap paralog-substitution analysis across selected species pairs. |


In [ ]:
PROJECT_DIR = Path(".")
CONFIG = load_samap_config(
    PROJECT_DIR / "config" / "samap_config.json"
)

FILES = species_files_from_config(
    CONFIG,
    workflow="celltype",
    project_dir=PROJECT_DIR,
)

MAP_DIR = PROJECT_DIR / CONFIG["paths"]["maps_dir"]

OUT = result_directories_from_config(
    CONFIG,
    workflow="celltype",
    project_dir=PROJECT_DIR,
)

SPECIES_ORDER = list(CONFIG["species"])
SPECIES_LABELS = {
    sid: info["name"]
    for sid, info in CONFIG["species"].items()
}

PARAMS = CONFIG["parameters"]


## Validate required inputs

Checks that all expected `.h5ad` files exist and contain the required broad-cell-type metadata, and verifies that every expected pairwise BLAST map is available before running SAMap.


In [ ]:
validate_h5ad_inputs(
    FILES,
    required_obs_columns={"cell_type"},
)

validate_blast_maps(
    MAP_DIR,
    SPECIES_ORDER,
)


## Load the original normalized species AnnData objects

Loads the species-specific normalized `.h5ad` files. These objects are kept because gene-expression overlays use the original within-species expression values rather than expression from the integrated SAMap object.


In [ ]:
species_adatas = {
    sid: sc.read_h5ad(path)
    for sid, path in FILES.items()
}


## Run SAM and multi-species SAMap

Runs SAM independently for each species and then uses SAMap to align the species in a shared cross-species manifold using expression structure together with the precomputed sequence-homology maps.


In [ ]:
sams = {}

for species_id, path in FILES.items():
    sam = SAM()
    sam.load_data(str(path))
    sam.preprocess_data()
    sam.run()
    sams[species_id] = sam

sm = SAMAP(
    sams=sams,
    f_maps=str(MAP_DIR),
)

with threadpool_limits(limits=PARAMS["thread_limit"]):
    sm.run(pairwise=True)

adata = sm.samap.adata
adata.write_h5ad(OUT["objects"] / "integrated.h5ad")


In [ ]:
validate_integrated_metadata(
    adata,
    required_columns={"species"},
)


## Combine species-specific cell-type metadata

Creates one common cell-type annotation column in the integrated object so broad cell types can be compared consistently across species.


In [ ]:
annotation_columns = {
    sid: f"{sid}_cell_type"
    for sid in FILES
}

adata = combine_species_metadata(
    adata,
    annotation_columns=annotation_columns,
    output_col="cell_type_combined",
)


## Plot species on the shared SAMap UMAP — combined

Shows all cells on the same integrated SAMap embedding and identifies each species by color/marker. This plot visualizes cross-species alignment; no gene-expression scaling is applied.


In [ ]:
plot_samap_species(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "samap_species_combined.png",
)


## Plot species on the shared SAMap UMAP — separate panels

Displays each species separately while preserving the same integrated SAMap coordinates, making it easier to see where each species contributes to the shared manifold.


In [ ]:
plot_samap_species_separate(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "samap_species_separate.png",
)


## Plot broad cell types on the shared SAMap UMAP — combined

Colors cells by broad cell type while retaining species-specific markers, allowing the same cell classes to be compared across species on one integrated embedding.


In [ ]:
species_markers = {
    "hu": "o",
    "mo": "^",
    "pi": "s",
    "ch": "D",
    "ze": "P",
    "ti": "X",
}

_, _, celltype_colors = plot_samap_celltypes_combined(
    adata,
    species_markers=species_markers,
    species_labels=SPECIES_LABELS,
    celltype_order=["Fibroblasts", "SMCs", "Endothelial", "Epithelial"],
    save_path=OUT["figures"] / "samap_celltypes_combined.png",
)


## Plot broad cell types on separate species UMAP panels

Shows the broad cell-type annotations one species at a time using the same integrated SAMap coordinates, which makes species-specific coverage and missing populations easier to inspect.


In [ ]:
plot_samap_celltypes_separate_species(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    celltype_order=["Fibroblasts", "SMCs", "Endothelial", "Epithelial"],
    celltype_colors=celltype_colors,
    save_path=OUT["figures"] / "samap_celltypes_separate.png",
)


## Check whether a gene of interest is present in each species

Checks each source AnnData object for the requested gene and reports the exact stored feature name. For configured species, the helper can also detect simple paralog forms such as `a/b` duplicates.


In [ ]:
gene_of_interest = "GENE_OF_INTEREST"

gene_presence = check_gene_presence_across_species(
    species_adatas,
    anchor_gene=gene_of_interest,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    paralog_species=("ze",),
)

gene_presence


## Plot a gene across species — separate panels

Uses the integrated SAMap coordinates for cell position but takes normalized expression from each original species AnnData object. Each gene/species panel is independently scaled to its own 99th-percentile expression value.


In [ ]:
plot_gene_across_species_panels(
    integrated_adata=adata,
    species_adatas=species_adatas,
    anchor_gene=gene_of_interest,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    paralog_species=("ze",),
    percentile=PARAMS["expression_percentile"],
    save_path=OUT["figures"] / "gene_separate_panels.png",
)


## Plot a gene across species — one combined UMAP

Places all species on one SAMap UMAP and rescales expression independently within each species so the 99th percentile equals 1. Multiple requested paralogs in one species are combined by taking the maximum expression per cell before scaling.


In [ ]:
plot_gene_combined_relative_expression(
    integrated_adata=adata,
    species_adatas=species_adatas,
    anchor_gene=gene_of_interest,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    paralog_species=("ze",),
    percentile=PARAMS["expression_percentile"],
    save_path=OUT["figures"] / "gene_combined_relative_expression.png",
)


## Cell-type mapping scores

Quantifies how strongly annotated cell populations from different species map to one another in the SAMap alignment.


In [ ]:
keys = {
    sid: "cell_type"
    for sid in FILES
}

best_scores, mapping_table = get_celltype_mapping_scores(
    sm,
    keys=keys,
    n_top=0,
)

best_scores.to_csv(OUT["tables"] / "best_mapping_scores.csv")
mapping_table.to_csv(OUT["tables"] / "pairwise_mapping_scores.csv")


## Fibroblast GenePairFinder + BLAST tables

Identifies cross-species gene pairs that support fibroblast-to-fibroblast alignment and attaches BLAST sequence-similarity statistics to each pair. These pairwise tables are the input for the downstream fibroblast gene-network analysis.


In [ ]:
gpf = GenePairFinder(sm, keys=keys)

fibroblast_labels = {
    sid: f"{sid}_Fibroblasts"
    for sid in FILES
}

all_pairs = run_all_pairwise_gene_pairs(
    gpf,
    celltype_labels=fibroblast_labels,
    maps_dir=MAP_DIR,
    species_order=SPECIES_ORDER,
    output_dir=OUT["tables"] / "fibroblast_gene_pairs",
)


## Fibroblast gene-marker graph

Builds a NetworkX graph in which nodes are species-specific genes and edges are GenePairFinder-supported cross-species gene relationships. Connected components represent groups of genes linked across one or more species.


In [ ]:
gene_graph = build_gene_pair_network(all_pairs)

component_df = summarize_connected_components(
    gene_graph,
    species_order=SPECIES_ORDER,
)

component_df = add_species_combo(
    component_df,
    species_order=SPECIES_ORDER,
)

component_df.to_csv(
    OUT["tables"] / "fibroblast_gene_network_components.csv",
    index=False,
)


## Count exact species-combination frequencies

Counts how many connected fibroblast gene-network components have each exact species-combination pattern and displays the 30 most common combinations.


In [ ]:
combo_counts = (
    component_df["species_combo"]
    .value_counts()
    .rename_axis("species_combo")
    .reset_index(name="n_components")
)

combo_counts.head(30)


## UpSet plot of species combinations

Summarizes which exact combinations of species occur in the connected fibroblast gene-network components, making broadly shared and lineage-restricted network patterns easier to compare.


In [ ]:
plot_species_upset(
    component_df,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    title="Species combinations of fibroblast gene networks",
    save_path=OUT["figures"] / "fibroblast_gene_network_species_upset.png",
)


## Save connected components by exact species combination

Groups connected components by their exact `species_combo` label and saves one CSV per combination for easier inspection of patterns such as mammal-only, fish-only, or broadly shared components.


In [ ]:
save_components_by_species_combo(
    component_df,
    OUT["tables"] / "species_combo_components",
)


## Plot an individual gene-network component

Draws one selected connected component from the fibroblast gene graph so the specific genes and cross-species edges within that component can be inspected visually.


In [ ]:
# Choose a component ID after inspecting component_df.
#
# plot_gene_network_component(
#     gene_graph,
#     component_df,
#     group_id=1,
#     save_path=OUT["figures"] / "gene_network_component.png",
# )


## Save run metadata

Records the key analysis parameters and installed package versions next to the results so the run can be reproduced later.


In [ ]:
save_run_metadata(
    OUT["metadata"] / "run_metadata.json",
    parameters=PARAMS,
    extra={
        "workflow": "celltype",
        "species": SPECIES_ORDER,
        "input_files": {
            sid: str(path)
            for sid, path in FILES.items()
        },
    },
)
